In [ ]:
import torch
from rdkit import Chem
from rdkit.Chem import rdmolops
import pandas as pd
import pickle
import numpy as np

def one_hot_encoding(value, choices):
    """生成 one-hot 编码"""
    one_hot = [0] * len(choices)
    if value in choices:
        one_hot[choices.index(value)] = 1
    return one_hot

def gaussian_basis_expansion(distance, r0_values, sigma=0.5):
    """基于高斯基底的距离扩展"""
    return np.exp(-(np.array(r0_values) - distance)**2 / (2 * sigma**2))

def get_reverse_edge_index(edge_index):
    row, col = edge_index
    num_edges = edge_index.shape[1]
    edge_dict = {(row[i].item(), col[i].item()): i for i in range(num_edges)}
    rev_edge_index = torch.zeros(num_edges, dtype=torch.long)
    for i in range(num_edges):
        rev_edge_index[i] = edge_dict[(col[i].item(), row[i].item())]
    return rev_edge_index

# 定义常见元素的电负性（以保罗-鲍尔电负性为例）
electronegativity_dict = {
    'H': 2.20, 'He': 0.00, 'Li': 0.98, 'Be': 1.57, 'B': 2.04, 'C': 2.55, 'N': 3.04,
    'O': 3.44, 'F': 3.98, 'Ne': 0.00, 'Na': 0.93, 'Mg': 1.31, 'Al': 1.61, 'Si': 1.90,
    'P': 2.19, 'S': 2.58, 'Cl': 3.16, 'Ar': 0.00, 'K': 0.82, 'Ca': 1.00, 'Sc': 1.36,
    'Ti': 1.54, 'V': 1.63, 'Cr': 1.66, 'Mn': 1.55, 'Fe': 1.83, 'Co': 1.88, 'Ni': 1.91,
    'Cu': 1.90, 'Zn': 1.65, 'Ga': 1.81, 'Ge': 2.01, 'As': 2.18, 'Se': 2.55, 'Br': 2.96,
    'Kr': 0.00, 'Rb': 0.82, 'Sr': 0.95, 'Y': 1.22, 'Zr': 1.33, 'Nb': 1.60, 'Mo': 1.62,
    'Tc': 1.90, 'Ru': 2.20, 'Rh': 2.28, 'Pd': 2.20, 'Ag': 1.93, 'Cd': 1.69, 'In': 1.78,
    'Sn': 1.96, 'Sb': 2.05, 'I': 2.66, 'Xe': 0.00, 'Cs': 0.79, 'Ba': 0.89, 'La': 1.10,
    'Ce': 1.12, 'Pr': 1.13, 'Nd': 1.14, 'Pm': 1.13, 'Sm': 1.17, 'Eu': 1.20, 'Gd': 1.20,
    'Tb': 1.23, 'Dy': 1.22, 'Ho': 1.23, 'Er': 1.24, 'Tm': 1.25, 'Yb': 1.10, 'Lu': 1.27,
    'Hf': 1.30, 'Ta': 1.50, 'W': 1.70, 'Re': 1.90, 'Os': 2.20, 'Ir': 2.20, 'Pt': 2.28,
    'Au': 2.54, 'Hg': 2.00, 'Tl': 1.62, 'Pb': 2.33, 'Bi': 2.02, 'Po': 2.00, 'At': 2.20,
    'Rn': 0.00, 'Fr': 0.70, 'Ra': 0.90, 'Ac': 1.10
}

def extract_atom_features(mol):
    """提取分子中每个原子的特征（包括电负性）"""
    atom_features = {}
    for atom in mol.GetAtoms():
        atom_symbol = atom.GetSymbol()
        # 获取电负性，若原子在字典中没有，则设为默认值
        electronegativity = electronegativity_dict.get(atom_symbol, 2.55)  # 默认为碳的电负性
        atom_features[atom.GetIdx()] = {'electronegativity': electronegativity}
    return atom_features

def bond_features(bond, distance_matrix, atom_idx1, atom_idx2, atom_features):
    """提取键的描述符"""
    
    # 键类型（one-hot 编码）
    bond_type = bond.GetBondType()
    bond_type_encoding = one_hot_encoding(bond_type, [Chem.rdchem.BondType.SINGLE,
                                                      Chem.rdchem.BondType.DOUBLE,
                                                      Chem.rdchem.BondType.TRIPLE,
                                                      Chem.rdchem.BondType.AROMATIC])
    
    # 是否在同一个环中（binary）
    same_ring = int(bond.IsInRing())

    # 图距离 (拓扑距离)
    graph_distance = min(int(distance_matrix[atom_idx1, atom_idx2]), 7)  # 限制距离最大为 7

    # 基于高斯基底的距离扩展
    r0_values = np.linspace(0, 4, 20)  # 在 0 到 4 之间取 20 个等间距的点
    expanded_distance = gaussian_basis_expansion(graph_distance, r0_values)
    
#     # 计算环的大小（如果是环键）
#     ring_size = bond.GetOwningMol().GetRingInfo().NumRings() if same_ring else 0
    
    # 获取两端原子的电负性差
    atom1 = bond.GetBeginAtom()
    atom2 = bond.GetEndAtom()

    # 获取电负性值
    electronegativity1 = atom_features[atom_idx1].get('electronegativity', 2.55)  # 默认为碳的电负性
    electronegativity2 = atom_features[atom_idx2].get('electronegativity', 2.55)
    
    # 计算电负性差异
    electronegativity_difference = abs(electronegativity1 - electronegativity2)
    
#     # 判断键是否是芳香键
#     is_aromatic = bond.GetIsAromatic()

    # 返回所有特征
    return bond_type_encoding + [same_ring, graph_distance, electronegativity_difference] + expanded_distance.tolist()


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    
    # 如果只有一个原子，则返回一个简单的图结构
    if mol.GetNumAtoms() == 1:
        edge_index = torch.tensor([[], []], dtype=torch.long)  # 空的边索引
        edge_attr = torch.tensor([], dtype=torch.float)  # 空的边特征
        rev_edge_index = torch.tensor([], dtype=torch.long)  # 空的反向边索引
        return edge_index, edge_attr, rev_edge_index
    
    # 获取拓扑距离矩阵
    distance_matrix = rdmolops.GetDistanceMatrix(mol)
    
    # 提取原子特征
    atom_features = extract_atom_features(mol)
    
    # 获取边索引和边特征
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.append([start, end])
        edge_index.append([end, start])
        
        # 提取更丰富的键描述符
        bond_features_start = bond_features(bond, distance_matrix, start, end, atom_features)
        bond_features_end = bond_features(bond, distance_matrix, end, start, atom_features)
        
        edge_attr.append(bond_features_start)
        edge_attr.append(bond_features_end)

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    
    rev_edge_index = get_reverse_edge_index(edge_index)
    
    return edge_index, edge_attr, rev_edge_index

def process_smiles_to_graph_pickle(csv_path, smiles_column='smiles', output_prefix='graph_data'):
    df = pd.read_csv(csv_path)

    edge_index_list = []
    edge_attr_list = []
    rev_edge_index_list = []

    for smiles in df[smiles_column]:
        edge_index, edge_attr, rev_edge_index = smiles_to_graph(smiles)
        if edge_index is None:
            edge_index_list.append(None)
            edge_attr_list.append(None)
            rev_edge_index_list.append(None)
        else:
            edge_index_list.append(edge_index.numpy())
            edge_attr_list.append(edge_attr.numpy())
            rev_edge_index_list.append(rev_edge_index.numpy())

    # 保存文件名
    base = os.path.splitext(os.path.basename(csv_path))[0]
    with open(f'{output_prefix}_edge_index.pkl', 'wb') as f:
        pickle.dump(edge_index_list, f)
    with open(f'{output_prefix}_edge_attr.pkl', 'wb') as f:
        pickle.dump(edge_attr_list, f)
    with open(f'{output_prefix}_rev_edge_index.pkl', 'wb') as f:
        pickle.dump(rev_edge_index_list, f)

    print(f"Graph data saved to {output_prefix}_*.pkl")

In [ ]:
import pandas as pd
import numpy as np
import pickle
from rdkit import Chem
import torch
from rdkit.Chem import rdchem

def extract_atom_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Invalid SMILES: {smiles}")
        return None
    
    atom_features = []
    
    # 定义原子类型（依据第一行的原子种类）
    atom_types = ['H', 'B', 'C', 'N', 'O', 'F', 'Si', 'Na', 'P', 'S', 'Cl', 'Se', 'Br', 'I', 'Ge', 'Sn', 'Te']
    
    # 定义杂化类型
    hybridizations = ['SP2', 'SP3', 'SP', 'S']
    
    # 定义范德华半径（单位：Å，未找到数据的元素用 None 代替）
    vdw_radius_dict = {
        'H': 1.2, 'B': 1.9, 'C': 1.7, 'N': 1.55, 'O': 1.52, 'F': 1.47, 
        'Si': 2.1, 'Na': 2.27, 'P': 1.8, 'S': 1.8, 'Cl': 1.75, 'Se': 1.9, 
        'Br': 1.85, 'I': 1.98, 'Ge': 2.11, 'Sn': 2.17, 'Te': 2.06
    }
    
    # 定义电负性（Pauling 标度，未找到数据的元素用 None 代替）
    electronegativity_dict = {
        'H': 2.20, 'B': 2.04, 'C': 2.55, 'N': 3.04, 'O': 3.44, 'F': 3.98, 
        'Si': 1.90, 'Na': 0.93, 'P': 2.19, 'S': 2.58, 'Cl': 3.16, 'Se': 2.55, 
        'Br': 2.96, 'I': 2.66, 'Ge': 2.01, 'Sn': 1.96, 'Te': 2.1
    }
    
    
    for atom in mol.GetAtoms():
        features = {}
        
        # Atom type one-hot encoding
        atom_type = atom.GetSymbol()
        features.update({f'atom_type_{at}': 1 if atom_type == at else 0 for at in atom_types})
        
        # Hybridization one-hot encoding
        hybridization = str(atom.GetHybridization())
        features.update({f'hybridization_{hyb}': 1 if hybridization == hyb else 0 for hyb in hybridizations})
        
        # Ring structure check
        features['is_cyclic'] = 1 if atom.IsInRing() else 0
        
        # Aromaticity check
        features['is_aromatic'] = 1 if atom.GetIsAromatic() else 0
        
        # Acceptor/Donor based on atom's implicit hydrogen count
        features['is_acceptor'] = 1 if atom.GetTotalDegree() - atom.GetNumImplicitHs() > 0 else 0
        features['is_donor'] = 1 if atom.GetNumImplicitHs() > 0 else 0
        
        # Valence (explicit or implicit)
        features['explicit_valence'] = atom.GetExplicitValence()
        features['implicit_valence'] = atom.GetImplicitValence()
        
        # Formal charge
        features['formal_charge'] = atom.GetFormalCharge()
        
        # Degree (simplified to just a single value)
        degree = atom.GetDegree()
        features['degree'] = degree
        
        # Hydrogen count
        features['total_H'] = atom.GetTotalNumHs()
        
        # Van der Waals radius (from the dictionary)
        features['vdw_radius'] = vdw_radius_dict.get(atom_type, 1.7)  # Default to C if unknown
        
        # Atomic number
        features['atomic_number'] = atom.GetAtomicNum()

        # Radical Electrons estimation
        features['radical_electrons'] = 1 if atom.GetNumRadicalElectrons() > 0 else 0
        
        # Electronegativity
        features['electronegativity'] = electronegativity_dict.get(atom_type, 2.55)  # Default to Carbon if unknown
        
        # Add the atom features to the list
        atom_features.append(features)
    
    return atom_features


def convert_features_to_tensor(atom_features):
    num_features = len(atom_features[0])
    num_atoms = len(atom_features)
    tensor = np.zeros((num_atoms, num_features))
    for i, feature in enumerate(atom_features):
        for j, (key, value) in enumerate(feature.items()):
            tensor[i, j] = value
    return tensor

def smiles_to_atom_feature_pickle(csv_path, smiles_column, output_pkl_path='atom_features.pkl'):
    """
    从CSV中读取SMILES并提取原子特征，保存为.pkl

    参数:
    - csv_path: CSV文件路径
    - smiles_column: 包含SMILES的列名
    - output_pkl_path: 输出pkl文件路径
    """
    df = pd.read_csv(csv_path)
    if smiles_column not in df.columns:
        raise ValueError(f"指定的列 {smiles_column} 不存在于文件中。")

    smiles_list = df[smiles_column].tolist()
    all_atom_features = []

    for smiles in smiles_list:
        try:
            atom_features = extract_atom_features(smiles)
            if atom_features is None:
                raise ValueError(f"Failed to extract atom features for: {smiles}")
            atom_tensor = convert_features_to_tensor(atom_features)
            all_atom_features.append(atom_tensor)
        except Exception as e:
            print(f"处理失败: {smiles}, 错误: {e}")

    with open(output_pkl_path, 'wb') as f:
        pickle.dump(all_atom_features, f)

    print(f"已成功提取并保存原子特征到 {output_pkl_path}")


In [ ]:
import pickle
import torch

# 1. 加载原始节点特征文件
with open('chromophore_atom_feature.pkl', 'rb') as f:
    node_features_list = pickle.load(f)

# 输出第一个图的形状
first_graph_shape = node_features_list[0].shape
print(f"第一个图的形状: {first_graph_shape}")

# 2. 确定每个图的节点数量
num_nodes_per_graph = [graph.shape[0] for graph in node_features_list]

# 3. 生成 batch_indices
# 这里生成的是每个分子的节点数量
batch_indices = torch.tensor(num_nodes_per_graph, dtype=torch.long)

# 输出前十个 batch_indices 的内容以检查
print(f"前十个 batch_indices 的内容: {batch_indices.numpy()}")

# 4. 保存 batch_indices 到新的 .pkl 文件
with open('batch_indices.pkl', 'wb') as f:
    pickle.dump(batch_indices.numpy(), f)  # 转换为 NumPy 数组以便保存

print("batch_indices 文件 'batch_indices.pkl' 已保存。")
